# Exploring decisions
It will help better understand those PDFs and do better chunking and single RAG

## Imports and paths

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pdfplumber # Great for inspecting PDF layout
from pathlib import Path
from tqdm.notebook import tqdm
from unidecode import unidecode
import re
import random
random.seed(42)
import json
from collections import Counter


# style for charts
sns.set_theme(style="whitegrid")

# define project paths based on your structure
BASE_DIR = Path('..') 
RAW_DATA_DIR = BASE_DIR / 'data' / '01_raw_pdfs'

EDA_DIR = BASE_DIR / "data" / "99_eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

print("RAW:", RAW_DATA_DIR.resolve())
print("EDA:", EDA_DIR.resolve())

## Definition - Text normalization + regex bank

In [ ]:
# --- DEFINING REGEX PATTERNS ---
import re
from unidecode import unidecode

def norm_pair(raw: str):
    """
    It prepares text for analysis - 2 versions
    Returns:
      raw_lower: lowercased original (keeps '§')
      ascii_norm: lower + unidecode (diacritics removed; '§' -> 'ss')
      I am doing it because of possibility of errors in PDFs
    """
    raw = raw or ""
    raw_lower = raw.lower()
    
    # unidecode sometimes messes up '§', so I handle it manually first
    ascii_norm = unidecode(raw_lower.replace("§", "ss")) 
    ascii_norm = re.sub(r"[ \t]+", " ", ascii_norm) # clean extra spaces and tabulators on one space
    return raw_lower, ascii_norm

def spaced_word_regex(word: str) -> str:
    # matches spaced-out headers like "o d o v o d n e n i e"
    # logic: insert optional space \s* between every letter
    return r"\s*".join(list(word))

# Applying spaced_word_regex to headers 
#  it matches both "odovodnenie" and "o d o v o d n e n i e"
RE_ODOV = re.compile(rf"{spaced_word_regex('odovodnenie')}|odôvodnenie", re.IGNORECASE)
RE_POUC = re.compile(rf"{spaced_word_regex('poucenie')}|poučenie", re.IGNORECASE)

# matches "rozhodol" even if letters are spaced out or split across lines
RE_VYROK = re.compile(rf"(?:takto\s*)?{spaced_word_regex('rozhodol')}|výrok|vyrok", re.IGNORECASE) 


# IMPORTANT: match both '§' (raw) and 'ss' (unidecode of '§')
PAR = r"(?:§|ss|par)\s*"

# problems with paragraph 301 in decisions - not always directly written 301
EXACT_301 = rf"(?:{PAR}301\b)|(?:\b301\b.{0,80}\b(obchodn|obchz|obchodn\w*\s+zakonnik))"
# is is sometimtes written as range "300 - 302" ("v sulade s paraggrafom 300 az 302 Obch. zakonnika ... adtd)
RANGE_300_302 = rf"(?:300\s*(?:až|-|do|a)\s*30[2-9])"

KEY = {
    # core
    "has_penalty": re.compile(r"\bzmluvn\w*\s+pokut\w*\b"),
    
    # merged 2 options of writing 301 in slovak decisions
    "has_301": re.compile(
    rf"{EXACT_301}|{RANGE_300_302}", 
    re.IGNORECASE | re.DOTALL),

    # civil noise
    "has_oz_545a": re.compile(PAR + r"545\s*a\b|" + PAR + r"545a\b"),
    "has_oz_544":  re.compile(PAR + r"544\b"),

    # moderation / proportionality signals
    "has_moder_trig": re.compile(r"\b(primeran\w*|neprimeran\w*|moder\w*|zniz\w*|poniz\w*|neprizna\w*|zamiet\w*)\b", re.IGNORECASE),

    # other grounds (to separate from moderation)
    "has_invalidity": re.compile(r"\b(neplatn\w*|neurcit\w*|dobr\w*\s+mrav\w*|poctiv\w*\s+obchodn\w*\s+styk\w*)\b", re.IGNORECASE),

    # interest confusion (rate != penalty)
    "has_interest": re.compile(r"\b(urok\w*|urok\s+z\s+omeskania|omeskan\w*)\b"),

    # candidates
    "has_percent": re.compile(r"(\d+(?:[.,]\d+)?)\s*%"),
    "has_money":   re.compile(r"(\d{1,3}(?:[ \.\u00A0]\d{3})*(?:,\d+)?|\d+(?:,\d+)?)\s*(eur|€|sk|sk\.)"),
    
    # civil noise filter
    "has_civil_code": re.compile(r"obciansk\w*\s+zakonnik", re.IGNORECASE),
}

## Load file registry

In [ ]:
records = []

# structure: 01_raw_pdfs / {case_folder} / {file}.pdf
pdf_files = list(RAW_DATA_DIR.rglob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files. Processing metadata...")

for pdf_path in tqdm(pdf_files):
    folder = pdf_path.parent
    
    # JSON: prefer exact match (file.pdf -> file.json)
    meta_path = pdf_path.with_suffix(".json")
    
    # Ak neexistuje presný názov, skúsime nájsť akýkoľvek JSON v priečinku (fallback)
    if meta_path.exists():
        json_files = [meta_path]
    else:
        json_files = list(folder.glob("*.json"))
    
    metadata = {}
    if json_files:
        try:
            with open(json_files[0], 'r', encoding='utf-8') as f:
                metadata = json.load(f)
        except Exception as e:
            print(f"Error reading JSON for {pdf_path.name}: {e}")

    # --- EXTRACTING FIELDS ---
    
    # 1. Court Name Logic 
    # first checking _csv_metadata - there is actual court which is reasoning the decision
    csv_meta = metadata.get('_csv_metadata', {})
    court_name = csv_meta.get('court') 
    
    if not court_name:
        court_info = metadata.get('sud', {})
        court_name = court_info.get('nazov', 'Unknown')
    
    # 2. Date
    raw_date = metadata.get('datumVydania', None)
    
    # 3. Case ID
    case_id = metadata.get('spisovaZnacka', folder.name)
    
    # 4. Decision Type
    decision_form = metadata.get('formaRozhodnutia', 'Unknown')
    if isinstance(decision_form, list):
        decision_form = decision_form[0] if decision_form else "Unknown"
        
    # 5. Oblast (Subject) ### NEW - Dôležité pre filtrovanie!
    subject_list = metadata.get('oblast', [])
    # Ak je to list, spojíme ho do stringu, napr. "Obchodné právo"
    subject = ", ".join(subject_list) if isinstance(subject_list, list) else str(subject_list)
    
    # 6. ECLI
    ecli = metadata.get('ecli', 'N/A')

    # Creating record
    record = {
        'filename': pdf_path.name,
        'rel_path': str(pdf_path.relative_to(BASE_DIR)), # Relative to project root
        'file_size_kb': round(pdf_path.stat().st_size / 1024, 2),
        'court': court_name, 
        'date_str': raw_date,
        'case_id': case_id,
        'type': decision_form,
        'subject': subject, ### NEW
        'ecli': ecli,
        'raw_metadata': metadata 
    }
    records.append(record)

# Convert to DataFrame
df = pd.DataFrame(records)

# Convert date column
df['date'] = pd.to_datetime(df['date_str'], format='%d.%m.%Y', errors='coerce', dayfirst=True)

# just sorting columns 
print(f"Loaded dataframe with {len(df)} rows.")
display(df[['case_id', 'court', 'subject', 'date', 'type']].head())

## Commercial scope report

In [ ]:
# --- 1.X Smart Commercial Scope Filter ---
# Goal: Ensure we only keep Commercial Law documents.
# We check two things: 
# 1. Metadata from the court (often contains errors).
# 2. Case ID (Spisova znacka) - this is reliable (e.g., 'Cob' is always commercial).

# 1. Get the main category from metadata
df["meta_oblast"] = df["raw_metadata"].apply(
    lambda m: (m.get("oblast") or ["Unknown"])[0] if isinstance(m, dict) else "Unknown"
)

# 2. Function to check if Case ID belongs to Commercial Registry
# Markers: 'cb', 'cob', 'cozm' (bills of exchange), 'cbi', 'cbs'
def is_trade_register(case_id):
    cid = str(case_id).lower()
    trade_markers = ['cb', 'cob', 'cozm', 'cbi', 'cbs'] 
    return any(marker in cid for marker in trade_markers)

df["is_trade_reg"] = df["case_id"].apply(is_trade_register)

# 3. Assign a Status to each document for clarity
def get_status(row):
    if row["meta_oblast"] == "Obchodné právo":
        return "VALID (Metadata OK)"
    elif row["is_trade_reg"] == True:
        return "RESCUED (Bad metadata, but Trade ID)"
    else:
        return "DROP (Civil/Other)"

df["filter_status"] = df.apply(get_status, axis=1)

# --- REPORTING ---

print("=== DATASET CLEANING REPORT ===")
print("Distribution of documents by status:")
print(df["filter_status"].value_counts())
print("-" * 40)

# Show what we are dropping (to be sure)
dropped_df = df[df["filter_status"] == "DROP (Civil/Other)"]
if not dropped_df.empty:
    print(f"Examples of removed documents ({len(dropped_df)} total):")
    # Displaying clean table without index for better readability
    display(dropped_df[["case_id", "court", "meta_oblast"]].head(5))
else:
    print("No documents to drop. Dataset is clean.")

# --- FILTERING ---
# We keep everything that is NOT marked as DROP
original_count = len(df)
df = df[df["filter_status"] != "DROP (Civil/Other)"].copy()
final_count = len(df)

print("=" * 40)
print(f"SUMMARY:")
print(f"Original files: {original_count}")
print(f"Removed files:  {original_count - final_count}")
print(f"Final dataset:  {final_count}")
print("=" * 40)

## Target courts report

In [ ]:
def court_tier(name: str) -> str:
    n = (name or "").lower().strip()

    # handle abbreviations like "KS Bratislava", "NS SR"
    if n.startswith("ns") or "najvyšší súd" in n:
        return "NS"
    if n.startswith("ks") or "krajský súd" in n:
        return "KS"
    if n.startswith("os") or "okresný súd" in n:
        return "OS"
    if n.startswith("ms") or "mestský súd" in n:
        return "MS"
    if n.startswith("us") or "ústavný súd" in n:
        return "US"
    return "OTHER"


df["court_tier"] = df["court"].apply(court_tier)
print(df["court_tier"].value_counts())
print("KS+NS ratio:", (df["court_tier"].isin(["KS","NS"])).mean())


## PDF quality - smoke test

In [ ]:
# --- 2.0 DETAILED PDF HEALTH CHECK ---
# Goal: Perform a deep audit of the PDF files to ensure text quality.
# We run 3 specific tests on every document sample:
# Test A: File Integrity (Can it be opened? Does it have pages?)
# Test B: Content Existence (Is there text, or is it a scanned image?)
# Test C: Encoding Quality (Is the text readable, or is it garbage symbols like '@#%')

def audit_pdf_health(pdf_path):
    """
    Analyzes the PDF and returns a detailed dictionary of metrics.
    We check the first 3 pages to get a representative sample.
    """
    stats = {
        "num_pages": 0,
        "char_count": 0,
        "valid_ratio": 0.0,  # Percentage of alphanumeric chars (a-z, 0-9)
        "status": "UNKNOWN",
        "snippet": ""
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # METRIC 1: Number of pages
            stats["num_pages"] = len(pdf.pages)
            
            if stats["num_pages"] == 0:
                stats["status"] = "FAILED: Empty File"
                return stats
            
            # Extract text from first 3 pages (or less if file is short)
            # We use 3 pages to skip potential cover sheets
            full_text = ""
            pages_to_check = min(3, stats["num_pages"])
            
            for i in range(pages_to_check):
                page_text = pdf.pages[i].extract_text()
                if page_text:
                    full_text += page_text + " "
            
            # METRIC 2: Total characters extracted
            stats["char_count"] = len(full_text)
            
            # TEST B: Content Existence
            if stats["char_count"] < 50: # Arbitrary threshold for "empty"
                stats["status"] = "FAILED: No Text (Scanned?)"
                return stats
            
            # METRIC 3: Valid Character Ratio (The "Garbage" Detector)
            # We count how many characters are letters or numbers.
            # If we extract "(CID:932) 0021", the ratio will be very low.
            valid_chars = sum(c.isalnum() for c in full_text)
            stats["valid_ratio"] = round(valid_chars / stats["char_count"], 3)
            
            stats["snippet"] = full_text[:100].replace('\n', ' ')

            # TEST C: Encoding Quality
            # If less than 40% of the text is alphanumeric, it is likely corrupted.
            if stats["valid_ratio"] < 0.40:
                stats["status"] = "FAILED: Garbage Encoding"
            else:
                stats["status"] = "PASSED (Healthy)"
                
            return stats

    except Exception as e:
        stats["status"] = f"CRITICAL ERROR: {str(e)}"
        return stats

# --- RUNNING THE AUDIT ---
print("Running detailed health check on sample documents...")

audit_data = []
# Taking a sample of 15 documents to be sure
sample_docs = df.head(15).copy()

for _, row in tqdm(sample_docs.iterrows(), total=len(sample_docs)):
    full_path = BASE_DIR / row['rel_path']
    
    # Run the audit function
    result = audit_pdf_health(full_path)
    
    # Add metadata for context
    result['filename'] = row['filename']
    result['case_id'] = row['case_id']
    audit_data.append(result)

# Create a DataFrame for the report
audit_df = pd.DataFrame(audit_data)

# Reorder columns for better readability
cols = ['filename', 'status', 'num_pages', 'char_count', 'valid_ratio', 'snippet']
audit_df = audit_df[cols]

# --- FINAL REPORT ---
print("\n=== PDF HEALTH AUDIT REPORT ===")
print(f"Total checked: {len(audit_df)}")
print("\nStatus Distribution:")
print(audit_df['status'].value_counts())

print("\nDetailed breakdown (Top 10):")
# We display the metrics clearly so anyone can see the proof
display(audit_df.head(10))

# Check if we have any failures
failures = audit_df[audit_df['status'].str.contains("FAILED")]
if not failures.empty:
    print(f"\nWARNING: Found {len(failures)} problematic files!")
    display(failures)
else:
    print("\nSUCCESS: All sampled files passed the integrity tests.")

## Text statistics

In [ ]:
def extract_full_text(pdf) -> str:
    # I am extracting text page by page
    # filter out None to avoid errors if a page is completely blank
    valid_text = [p.extract_text() for p in pdf.pages if p.extract_text()]
    # join with double newlines to keep paragraphs separated
    return "\n\n".join(valid_text)

rows = []
print(f"Processing {len(df)} files...")

for _, row in tqdm(df.iterrows(), total=len(df)):
    path = BASE_DIR / row["rel_path"]
    
    # safe initialization: start with basic info
    out = {"rel_path": row["rel_path"]}

    # IMPORTANT: Pre-fill all counting columns with 0.
    # If the PDF crashes later, I still want '0' in my table, not 'NaN' (empty).
    for k in KEY.keys():
        out[k] = 0
    out["has_odovodnenie"] = 0
    out["has_poucenie"] = 0
    out["has_vyrok"] = 0
    out["error"] = None # placeholder for errors

    try:
        with pdfplumber.open(path) as pdf:
            out["page_count"] = len(pdf.pages)

            t = extract_full_text(pdf)
            out["char_count"] = len(t)
            # simple estimation: 1 token is roughly 4 characters
            out["est_tokens"] = int(out["char_count"] / 4)

            # normalizing text for regex search
            # raw_lower = for reading, ascii_norm = for searching (removes fada/makcen)
            raw_lower, ascii_norm = norm_pair(t)

            # counting keywords from my regex bank
            for k, rx in KEY.items():
                out[k] = len(rx.findall(ascii_norm))

            # checking for document structure (headers)
            out["has_odovodnenie"] = int(bool(RE_ODOV.search(ascii_norm)))
            out["has_poucenie"] = int(bool(RE_POUC.search(ascii_norm)))
            out["has_vyrok"] = int(bool(RE_VYROK.search(ascii_norm)))

            # calculating paragraph statistics to see if text is chunk-ready
            # splitting by empty lines
            pars = [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]
            par_lens = [int(len(p)/4) for p in pars] 
            
            out["n_pars"] = len(pars)
            out["par_max_tokens"] = max(par_lens) if par_lens else 0
            # getting the 90th percentile to see how long the big paragraphs are
            out["par_p90_tokens"] = int(np.percentile(par_lens, 90)) if len(par_lens) >= 2 else (par_lens[0] if par_lens else 0)

    except Exception as e:
        # if file is corrupted, set counts to 0 and save the error message
        out["page_count"] = 0
        out["char_count"] = 0
        out["est_tokens"] = 0
        out["error"] = str(e)

    rows.append(out)

df_feat = pd.DataFrame(rows)

# cleanup: remove old feature columns before merging to avoid duplicates
feature_cols = [c for c in df_feat.columns if c != "rel_path"]
df = df.drop(columns=[c for c in feature_cols if c in df.columns], errors="ignore")

# joining the new features to the main table
df = df.merge(df_feat, on="rel_path", how="left")

# checking the results
df[["filename", "page_count","char_count","est_tokens","has_penalty","has_301","error"]]

### Finding incomplete documents
Goal: I want to see only the files that are missing something important.
A valid judgment must have 3 parts: Verdict (Vyrok), Reasoning (Odovodnenie), Instruction (Poucenie).


In [ ]:
# 1. Create a filter for "Bad" documents
# If any of these counts is 0, the document is incomplete.
# logic: (No Verdict) OR (No Reasoning) OR (No Instruction)
bad_docs = df[
    (df["has_vyrok"] == 0) | 
    (df["has_odovodnenie"] == 0) | 
    (df["has_poucenie"] == 0)
]

print("--- DATA QUALITY REPORT ---")
print(f"Total documents: {len(df)}")
print(f"Incomplete documents found: {len(bad_docs)}")

# 2. Calculate percentage
# If this number is high (e.g., over 20%), I might need to improve my Regex or check the source.
if len(df) > 0:
    percent_bad = (len(bad_docs) / len(df)) * 100
    print(f"Percentage of incomplete files: {percent_bad:.1f}%")

# 3. Show the problematic files
# I want to see WHICH part is missing.
# 0 = Missing (BAD), 1 = Present (GOOD)
print("\n--- LIST OF PROBLEMATIC FILES ---")
cols = ["filename", "has_vyrok", "has_odovodnenie", "has_poucenie"]

if not bad_docs.empty:
    display(bad_docs[cols])
else:
    print("No incomplete documents were found. All files define the structure correctly.")

## Visualizing the Dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribution of tokens (how expensive will the LLM be?)
sns.histplot(df['est_tokens'], bins=20, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribution of Estimated Tokens')
axes[0].set_xlabel('Tokens')
axes[0].axvline(df['est_tokens'].mean(), color='red', linestyle='--', label='Mean')

# 2. distribution of years (temporal coverage)
if df['date'].notna().sum() > 0:
    df["year"] = df["date"].dt.year
    year_counts = df["year"].dropna().astype(int).value_counts().sort_index()
    
    sns.barplot(x=year_counts.index.astype(str), y=year_counts.values, ax=axes[1])
    axes[1].set_title("Decisions by year")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Count")

# 3. top courts (Where does data come from?)
top_courts = df['court'].value_counts().head(10)
sns.barplot(
    x=top_courts.values,
    y=top_courts.index,
    hue=top_courts.index,       
    palette='magma',
    ax=axes[2],
    legend=False             
)
axes[2].set_title('Top 10 Courts')

plt.tight_layout()
plt.show()


## Physical layout inspection
I picked one random file to inspect the raw data structure before processing.
* **Text check:** Verified that `pdfplumber` extracts readable text (it is not a scanned image).
* **Coordinate check:** I printed the bounding boxes (`bbox`) of the first 20 words.
* I need to see the `top` values. If the "Court Name" and "File ID" are always at `top < 100`, I can confirm that they are headers. This helps me verify if I can rely on position-based cleaning or just regex.

In [ ]:
# choose real document from df
sample_row = df.sample(1, random_state=42).iloc[0]
path = BASE_DIR / sample_row['rel_path']

with pdfplumber.open(path) as pdf:
    print(f"Analyzing: {sample_row['filename']}")
    print(f"Total pages: {len(pdf.pages)}")

    p1 = pdf.pages[0]
    text = p1.extract_text() or ""
    print("\n--- PAGE 1 (first 800 chars) ---")
    print(text[:800])

    # fast layout sanity (how much words and their bbox)
    words = p1.extract_words()[:20]
    print("\n--- FIRST 20 WORD BOXES ---")
    for w in words:
        print(w["text"], (w["x0"], w["top"], w["x1"], w["bottom"]))


## Contract type triggers

In [ ]:
# --- 6.0 CONTRACT TYPE IDENTIFICATION (EDA) ---
# Goal: Estimate the distribution of contract types in the dataset.
# We use ASCII regexes because we perform search on unidecoded text (to avoid accent errors).

from unidecode import unidecode
from collections import Counter

# Defining keywords without accents (č -> c, ž -> z, á -> a)
CONTRACT_RX = {
    "sale_purchase": re.compile(r"\bkupn\w+\s+zmluv\w+", re.IGNORECASE), # kupna zmluva
    "work_contract": re.compile(r"\bzmluv\w+\s+o\s+diel\w+", re.IGNORECASE), # zmluva o dielo
    "lease": re.compile(r"\bnajomn\w+\s+zmluv\w+", re.IGNORECASE), # najomna zmluva
    "loan_credit": re.compile(r"\b(uverov\w+|zmluv\w+\s+o\s+uvere|pozick\w+)", re.IGNORECASE), # uverova zmluva, pozicka (fixed: removed generic 'uver' to avoid 'uverejnenie')
    "leasing": re.compile(r"\bleasing\w*|lizing\w*", re.IGNORECASE), # leasing is common in commercial law
    "mandate": re.compile(r"\bmandatn\w+\s+zmluv\w+|\bprikazn\w+\s+zmluv\w+", re.IGNORECASE),
    "agency": re.compile(r"\bsprostredkovate[l]\w+\s+zmluv\w+", re.IGNORECASE), # sprostredkovatelska
    "insurance": re.compile(r"\bpoistn\w+\s+zmluv\w+", re.IGNORECASE),
}

print("Running contract type analysis on a sample...")

# We take a larger sample for better stats
sample_size = min(50, len(df))
sample = df.sample(sample_size, random_state=42)
hits = Counter()

# Iterate over the sample
for _, r in tqdm(sample.iterrows(), total=sample_size):
    path = BASE_DIR / r["rel_path"]
    try:
        with pdfplumber.open(path) as pdf:
            # Quick text extraction
            text = "\n".join([(p.extract_text() or "") for p in pdf.pages])
            
        # Normalize: lower + remove accents (pôžička -> pozicka)
        text_norm = unidecode(text).lower()
        
        # Check patterns
        found_any = False
        for k, rx in CONTRACT_RX.items():
            if rx.search(text_norm):
                hits[k] += 1
                found_any = True
        
        if not found_any:
            hits["unknown/other"] += 1
            
    except Exception as e:
        hits["error"] += 1

# Display results
print(f"\n--- Contract Types Distribution (Sample of {sample_size}) ---")
ct_df = pd.DataFrame(hits.most_common(), columns=["contract_type", "count"])
ct_df["percentage"] = (ct_df["count"] / sample_size) * 100
display(ct_df)

Exploratory analysis confirmed that the dataset contains a representative sample of commercial disputes. We identified the occurrence of key institutes (Section 301 of the Commercial Code) across different types of contractual relationships (purchase, for work, credit), which validates the suitability of the dataset for the task of information extraction.


## Does decisions talk about contractual penalty

In [ ]:
print(df["has_301"].gt(0).sum(), "docs mention §301")
print(df["has_penalty"].gt(0).sum(), "docs mention zmluvná pokuta")
# print(df["has_oz_545a"].gt(0).sum(), "docs mention OZ §545a")
print(df["has_oz_544"].gt(0).sum(), "docs mention OZ §544")

* **§ 544 OZ (Civil Code)**: *Creates the contractual penalty clause.* It says when a contractual penalty is **valid** — it must be **in writing** and the **amount (or a clear calculation method)** must be agreed. → Question: **“Does the penalty clause legally exist / is it enforceable?”**

* **§ 301 ObchZ (Commercial Code)**: *Business “moderation” rule.* In **commercial (business/B2B)** contracts, the court can also **reduce an excessive penalty**. → Question: **“In a commercial relationship, should the court cut the penalty down?”**


## Better categories

In [ ]:
def label(row):
    #  must check for §301 FIRST. 
    # (Commercial courts cite §544 for validity too, so we cannot discard it early).
    
    # 4. Civil/Consumer noise 
    # If no §301 was found above, and i see consumer laws (§545a), it is noise.
    if row["has_oz_545a"] > 0:
        return "4_CIVIL_NOISE_LIKELY"

    # 1. Gold standard: Penalty + §301 + keywords (e.g. "neprimerana", "znizil")
    if row["has_penalty"] > 0 and (row["has_301"] > 0) and (row["has_moder_trig"] > 0):
        return "1_GOLD_MODERATION"

    # 2. Silver: Penalty + §301 (strong candidate, maybe implied moderation)
    if row["has_penalty"] > 0 and row["has_301"] > 0:
        return "2_STRONG_CANDIDATE"

    # 3. Bronze: Penalty mentioned, but NO §301 (good negative samples)
    if row["has_penalty"] > 0:
        return "3_PENALTY_NO_MODERATION"



    return "5_IRRELEVANT"

df["eda_label"] = df.apply(label, axis=1)

print("--- Thesis relevance groups ---")
print(df["eda_label"].value_counts().sort_index())


In [ ]:
# --- INSPECT NON-GOLD CASES ---
# Goal: Check why those  documents didnt make it to "Gold".
# do i miss something or are the data correct?

# 1. Filtering everything that is NOT Gold
non_gold_df = df[df["eda_label"] != "1_GOLD_MODERATION"].copy()

print(f"Number of docs which are not in gold {len(non_gold_df)} \n")

# 2. Group by the label they got assigned
categories = sorted(non_gold_df["eda_label"].unique())

for cat in categories:
    print(f"\n{'='*20}")
    print(f"Categoory: {cat}")
    print(f"{'='*20}")
    
    # Get files in this category
    subset = non_gold_df[non_gold_df["eda_label"] == cat]
    print(f"Number of files: {len(subset)}")
    print("Showing first 5 for review:")
    
    # prinitnig names of files
    for i, row in subset.head(5).iterrows():
        print(f" - {row['filename']} (Court: {row.get('court', 'N/A')})")



## Applied vs not applied

In [ ]:
# Noote - this is only heuristic to guide manual audit sampling - very unprecise
RE_REDUCED = re.compile(r"\bzn[ií]žil\b|\bzni[zž]il\b|\bznížen\w*|\bznizen\w*", re.IGNORECASE)
RE_NOT_REDUCED = re.compile(r"neprist[uú]pil\w*\s+k\s+zni[zž]eniu|nebolo\s+potrebn\w*\s+.*modera|nevyhovel\w*\s+.*modera", re.IGNORECASE)

# We approximate using existing counts (full-text regex was counted in ONE PASS)

df["outcome_proxy"] = "UNCLEAR"
df.loc[(df["has_301"]>0) & (df["has_moder_trig"]>0), "outcome_proxy"] = "DISCUSSED"


df["outcome_proxy"].value_counts()


The final classification of the outcome of the proceedings (whether there was moderation) was not performed using regular expressions due to their inability to distinguish between the participant's proposal and the court's ruling. This task was left to the LLM model in the extraction phase.\

 It is because there can be something like this: "Žalovaný vo svojom vyjadrení navrhol, aby súd pokutu znížil (§ 301), avšak súd tomuto návrhu nevyhovel."


## Structural integrity check (the "section splitter" test)

In [ ]:

print("Odôvodnenie coverage:", (df["has_odovodnenie"] == 1).mean())
print("Poučenie coverage:", (df["has_poucenie"] == 1).mean())
print("Výrok coverage:", (df["has_vyrok"] == 1).mean())

# show the worst cases ( a littbe bit of text or damaged layout)
# df.sort_values(["has_odovodnenie","char_count"]).head(20)[
#     ["filename","court","case_id","page_count","char_count","has_odovodnenie","has_poucenie","has_vyrok","eda_label"]
# ]


## Chunk readiness: paragraph and long paragraphs

In [ ]:
print(df[["n_pars","par_p90_tokens","par_max_tokens"]].describe().round(1))

# how much documents will need token fallback? (> 1000 tokens per paragraph)
fallback_rate = (df["par_max_tokens"] > 1000).mean()
print("Docs needing token-fallback (par_max_tokens>1000):", round(fallback_rate, 3))

plt.figure(figsize=(8,4))
sns.histplot(df["par_max_tokens"], bins=30)
plt.title("Max paragraph length (token-ish)")
plt.show()


Paragraph length analysis on a sample dataset showed an extremely high level of text saturation  - up to 96.1% of documents containing text blocks exceeding 1000 tokens (average 1285 tokens).

## Interest vs penalty confusion

In [ ]:
# (proximity-based sanity on a small sample)
WINDOW = 250  # chars
sample = df.sample(min(25, len(df)), random_state=42)

def proximity_counts(text: str):
    if not text:
        return (0,0)
    t = text
    a = unidecode(t).lower()
    pct_positions = [m.start() for m in re.finditer(r"\d+(?:[.,]\d+)?\s*%", t)]
    penalty_positions = [m.start() for m in re.finditer(r"zmluvn\w*\s+pokut\w*", a)]
    interest_positions = [m.start() for m in re.finditer(r"(urok|úrok|omeskan|omeškan)", a)]

    def near(pos_list, anchor_list):
        for p in pos_list:
            if any(abs(p - q) <= WINDOW for q in anchor_list):
                return True
        return False

    return int(near(pct_positions, penalty_positions)), int(near(pct_positions, interest_positions))

rows = []
for _, r in sample.iterrows():
    path = BASE_DIR / r["rel_path"]
    with pdfplumber.open(path) as pdf:
        text = "\n\n".join([(p.extract_text() or "") for p in pdf.pages])
    near_penalty, near_interest = proximity_counts(text)
    rows.append({"filename": r["filename"], "near_penalty_pct": near_penalty, "near_interest_pct": near_interest})

prox = pd.DataFrame(rows)
prox.mean(numeric_only=True)


This code measures the "text confusion level." It scans the documents to see if the words "Penalty", "Interest", and the "%" symbol appear right next to each other (within 250 characters).

It proves that I cannot use simple search tools (Regex). Because the numbers are mixed together, a simple tool would mistake the Interest rate for the Penalty rate. This confirms that I need a smart AI (LLM) to understand the context.

## Manual audit

In [ ]:
OUT = BASE_DIR / "data" / "eda_outputs"
OUT.mkdir(parents=True, exist_ok=True)

def stratified_sample(df_, n=10):
    out = []
    for label, g in df_.groupby("eda_label"):
        out.append(g.sample(min(n, len(g)), random_state=42))
    return pd.concat(out).sample(frac=1, random_state=42)

audit = stratified_sample(df, n=8)[
    ["filename","rel_path","court","case_id","date","type","eda_label",
     "court_tier",
     "has_penalty","has_301","has_moder_trig","has_invalidity",
     "has_percent","has_money","has_interest",
     "page_count","char_count","est_tokens"]
].copy()

audit["MAN_is_trade_meta"] = ""   # should be yes (Obchodné právo)
audit["MAN_really_301"] = ""      # yes/no/unclear
audit["MAN_reasonableness_discussed"] = ""  # yes/no
audit["MAN_outcome"] = ""         # reduced / upheld / denied / unclear

audit.to_csv(OUT/"manual_audit_sample.csv", index=False)
audit.head()


# Occurences of meststky sud or okresny sud ?

In [ ]:
df[df["court_tier"].isin(["OS","MS"])][["court","case_id","type","date","rel_path"]].head(30)


## 301 paragraph - real occurences
 showing short context around the FIRST §301 match and highlight it

In [ ]:
pd.set_option("display.max_colwidth", None)   # show full strings
pd.set_option("display.width", 200)           # wider display
pd.set_option("display.max_columns", 20)

def first_match_context(rx, text: str, window=140):
    raw_lower, ascii_norm = norm_pair(text)
    m = rx.search(ascii_norm)
    if not m:
        return None
    s, e = m.span()
    left = ascii_norm[max(0, s-window):s]
    mid  = ascii_norm[s:e]
    right = ascii_norm[e:min(len(ascii_norm), e+window)]
    left = re.sub(r"\s+", " ", left).strip()
    mid  = re.sub(r"\s+", " ", mid).strip()
    right = re.sub(r"\s+", " ", right).strip()
    return f"{left} >>>{mid}<<< {right}"

# sample docs with has_301>0
sample = df[df["has_301"] > 0].sample(
    min(10, (df["has_301"] > 0).sum()),
    random_state=42
)

rows = []
for _, r in sample.iterrows():
    with pdfplumber.open(BASE_DIR / r["rel_path"]) as pdf:
        t = extract_full_text(pdf)
    ctx = first_match_context(KEY["has_301"], t)
    rows.append({
        "case_id": r["case_id"],
        "court": r["court"],
        "type": r["type"],
        "ctx": ctx
    })

ctx_df = pd.DataFrame(rows)

# 1) table view
display(ctx_df)

# 2) readable view (no truncation, easier to read)
for i, row in ctx_df.iterrows():
    print("="*120)
    print(f"{i+1}) {row['case_id']} | {row['court']} | {row['type']}")
    print(row["ctx"])


In [29]:
r = sample.iloc[0]
with pdfplumber.open(BASE_DIR / r["rel_path"]) as pdf:
    t = extract_full_text(pdf)

raw_lower, ascii_norm = norm_pair(t)
m = KEY["has_301"].search(ascii_norm)

print("CASE:", r["case_id"], "|", r["court"])
print("FOUND:", bool(m))
if m:
    print("MATCH:", m.group(0))
    print("SPAN:", m.span())
    print("CTX:", first_match_context(KEY["has_301"], t))


CASE: 16Cob/78/2013 | KS Trenčín
FOUND: True
MATCH: ss 301
SPAN: (24895, 24901)
CTX: a ziadna skoda a jemu zalovanemu nebolo preukazane porusenie zakona c. 82/1994 z.z. o statnych hmotnych rezervach. moderacne pravo v zmysle >>>ss 301<<< ob.z. prislucha sudu a spociva v tom, ze sud moze rozhodnut, ze zmluvnu pokutu neprizna vo vyske ako bola stranami dohodnuta, aj ked verite


# Final of EDA
SAVIN RESULTS FOR NEXT PIPELINE STEP 

save dataframe to CSV, so we can read it in notebook 02_process_text

i am saving only needed columns


In [30]:
EDA_OUT = BASE_DIR / "data" / "99_eda" / "outputs"
EDA_OUT.mkdir(parents=True, exist_ok=True)

# 1) Save the main feature table (but only columns I really need)

cols_to_save = [
    "filename", "rel_path", "case_id", "court", "date",
    "eda_label", "est_tokens"  # add more later if I need them
]

df_out_csv = EDA_OUT / "eda_table.csv"
df[cols_to_save].to_csv(df_out_csv, index=False)

try:
    df_out_parquet = EDA_OUT / "eda_table.parquet"
    df[cols_to_save].to_parquet(df_out_parquet, index=False)
except Exception as e:
    print("Parquet export skipped (missing dependency):", e)

print("Saved:", df_out_csv)

# 2) Save small machine-readable summary (for quick checks in next notebooks)
summary = {
    "n_raw_pdfs": int(len(list(RAW_DATA_DIR.rglob("*.pdf")))),
    "n_after_filter": int(len(df)),
    "court_tier_counts": df["court_tier"].value_counts().to_dict() if "court_tier" in df.columns else {},
    "eda_label_counts": df["eda_label"].value_counts().to_dict() if "eda_label" in df.columns else {},
    "has_301_docs": int(df["has_301"].gt(0).sum()) if "has_301" in df.columns else None,
    "has_penalty_docs": int(df["has_penalty"].gt(0).sum()) if "has_penalty" in df.columns else None,
    "avg_est_tokens": float(df["est_tokens"].mean()) if "est_tokens" in df.columns else None,
    "p95_est_tokens": float(df["est_tokens"].quantile(0.95)) if "est_tokens" in df.columns else None,
    "fallback_rate_par_max_gt_1000": float((df["par_max_tokens"] > 1000).mean()) if "par_max_tokens" in df.columns else None,
}

summary_path = EDA_OUT / "eda_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:", summary_path)

# 3) Save  regex patterns description (so it is documented)
regex_info = {
    "notes": "Patterns are used on ascii_norm (lower + unidecode + §->ss).",
    "PAR": PAR,
    "EXACT_301": EXACT_301,
    "RANGE_300_302": RANGE_300_302,
    "KEY_names": list(KEY.keys()),
}

regex_path = EDA_OUT / "regex_bank_info.json"
with open(regex_path, "w", encoding="utf-8") as f:
    json.dump(regex_info, f, ensure_ascii=False, indent=2)

print("Saved:", regex_path)


Parquet export skipped (missing dependency): Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
Saved: ../data/99_eda/outputs/eda_table.csv
Saved: ../data/99_eda/outputs/eda_summary.json
Saved: ../data/99_eda/outputs/regex_bank_info.json
